# Diarization (NeMo Nemotron-3-Diarization)
Who speaks when, for the wavs of `data/raw_audios/` that `pipeline_paths.py` picks (`AUDIOS`). Diarization only: the chunks are made in `02_chunk_transcribe`.

Needs NeMo: the project's `.venv` on the Mac has it, `.venv-denoise` on the DGX Spark does not.

**Per audio**, in `outputs/diarization/<name>/nemotron/` of the run folder (`RUN` in `pipeline_paths.py`: the project root, or `runs/<RUN>/`):
- `<name>.rttm`: the model's turns (`start`, `duration`, `speaker`)
- `stats.json`: speech / overlap / silence, and per speaker: seconds, share of the speech, turns, median turn length
- `timeline.png`: who speaks when

The model needs 16 kHz mono, so a temporary 16 kHz copy is made and deleted after the run.
Audios that already have an RTTM are not run through the model again unless `FORCE = True`. Their stats and timeline are always rebuilt (that takes a moment).

In [ ]:
import json
import math
import os
import subprocess
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from nemo.collections.asr.models import SortformerEncLabelModel

import importlib
import pipeline_paths as P
importlib.reload(P)   # a change of RUN in pipeline_paths.py counts without restarting the kernel

root = P.ROOT
out_root = P.OUTPUTS
print(P.describe())
wavs = P.audio_files()
for i, p in enumerate(wavs):
    print(i, p.name)

In [ ]:
ONLY = None    # None = every audio; or a list of indices from the list above, e.g. [3, 5]
FORCE = False  # True = run the model again even if the audio already has an RTTM

In [ ]:
diarizer = SortformerEncLabelModel.from_pretrained("nvidia/Nemotron-3-Diarization")
diarizer.eval()

# streaming settings, in 80 ms frames (340 = ~27 s chunks)
m = diarizer.sortformer_modules
m.chunk_len = 340
m.chunk_right_context = 40
m.fifo_len = 40
m.spkcache_update_period = 300
m.spkcache_len = 264
diarizer._check_streaming_parameters()

In [ ]:
RES = 100  # stats grid: 10 ms


def diarize(audio_path, tmp_dir):
    """(start, end, speaker) turns. The model gets a temporary 16 kHz mono copy."""
    wav16 = tmp_dir / "audio_16k_mono.tmp.wav"
    try:
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(audio_path),
                        "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", str(wav16)], check=True)
        segments = diarizer.diarize(audio=[str(wav16)], batch_size=1)
    finally:
        wav16.unlink(missing_ok=True)
    return sorted((float(s), float(e), spk) for s, e, spk in (line.split() for line in segments[0]))


def write_rttm(turns, name, path):
    file_id = "_".join(name.split())  # RTTM fields are separated by spaces: no spaces in the id
    tmp = path.with_suffix(".rttm.tmp")
    with open(tmp, "w") as f:
        for s, e, spk in turns:
            f.write(f"SPEAKER {file_id} 1 {s:.3f} {e - s:.3f} <NA> <NA> {spk} <NA> <NA>\n")
    os.replace(tmp, path)  # an interrupted run can't leave half an RTTM


def read_rttm(path):
    turns = []
    for line in path.read_text().splitlines():
        p = line.split()  # counted from the end, so an id with spaces still works
        s = float(p[-7])
        turns.append((s, s + float(p[-6]), p[-3]))
    return sorted(turns)


def diarization_stats(audio_name, total_sec, turns):
    speakers = sorted({spk for *_, spk in turns})
    n = math.ceil(total_sec * RES)
    active = np.zeros((len(speakers), n), dtype=bool)
    for s, e, spk in turns:
        active[speakers.index(spk), max(0, round(s * RES)):min(n, round(e * RES))] = True
    count = active.sum(axis=0)
    speech, overlap = int((count >= 1).sum()), int((count >= 2).sum())

    per_speaker = {}
    for k, spk in enumerate(speakers):
        turn_len = [e - s for s, e, x in turns if x == spk]
        per_speaker[spk] = {
            "seconds": round(active[k].sum() / RES, 2),
            "share_of_speech_pct": round(100 * active[k].sum() / max(speech, 1), 1),  # overlap counts for both
            "turns": len(turn_len),
            "median_turn_sec": round(float(np.median(turn_len)), 2),
        }
    return {
        "audio": audio_name,
        "total_seconds": round(total_sec, 2),
        "speakers": len(speakers),
        "turns": len(turns),
        "speech_seconds": round(speech / RES, 2),
        "speech_pct": round(100 * speech / n, 1),
        "overlap_seconds": round(overlap / RES, 2),
        "overlap_pct": round(100 * overlap / n, 1),
        "silence_seconds": round((n - speech) / RES, 2),
        "silence_pct": round(100 * (n - speech) / n, 1),
        "per_speaker": per_speaker,
    }


def plot_timeline(turns, title, path):
    speakers = sorted({spk for *_, spk in turns})
    fig, ax = plt.subplots(figsize=(12, 0.6 * len(speakers) + 1))
    for row, spk in enumerate(speakers):
        ax.broken_barh([(s / 60, (e - s) / 60) for s, e, x in turns if x == spk], (row - 0.4, 0.8))
    ax.set_yticks(range(len(speakers)), speakers)
    ax.set_xlabel("min")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=100)
    plt.close(fig)


def process(audio_path):
    name = audio_path.stem
    out_dir = out_root / "diarization" / name / "nemotron"
    out_dir.mkdir(parents=True, exist_ok=True)
    rttm_path = out_dir / f"{name}.rttm"

    if rttm_path.exists() and not FORCE:
        turns = read_rttm(rttm_path)
        print(f"  {len(turns)} saved turns")
    else:
        turns = diarize(audio_path, out_dir)
        write_rttm(turns, name, rttm_path)

    stats = diarization_stats(audio_path.name, sf.info(str(audio_path)).duration, turns)
    (out_dir / "stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
    plot_timeline(turns, audio_path.name, out_dir / "timeline.png")
    print(f"  {stats['speakers']} speakers, {stats['turns']} turns, speech {stats['speech_pct']}%, "
          f"overlap {stats['overlap_pct']}%, silence {stats['silence_pct']}%")
    return stats

## Run

In [ ]:
selected = wavs if ONLY is None else [wavs[i] for i in ONLY]
failed = {}

for n, audio_path in enumerate(selected, 1):
    print(f"[{n}/{len(selected)}] {audio_path.name}")
    try:
        process(audio_path)
    except Exception:
        failed[audio_path.name] = traceback.format_exc()
        print("  FAILED (continuing)")

print(f"\ndone: {len(selected) - len(failed)}, failed: {len(failed)}")
for name, tb in failed.items():
    print("\n" + name + "\n" + tb)

## Stats of every audio
Read from the `stats.json` files, so it also works after a restart. `main%` = share of the speech of the speaker who talks most.

In [ ]:
all_stats = [json.loads(p.read_text(encoding="utf-8"))
             for p in sorted((out_root / "diarization").glob("*/nemotron/stats.json"))]

print(f"{'audio':55} {'min':>6} {'spk':>4} {'turns':>6} {'speech%':>8} {'overlap%':>9} {'silence%':>9} {'main%':>6}")
for s in all_stats:
    main = max((v["share_of_speech_pct"] for v in s["per_speaker"].values()), default=0)
    print(f"{s['audio'][:55]:55} {s['total_seconds'] / 60:6.1f} {s['speakers']:4d} {s['turns']:6d} "
          f"{s['speech_pct']:8.1f} {s['overlap_pct']:9.1f} {s['silence_pct']:9.1f} {main:6.1f}")
total = sum(s["total_seconds"] for s in all_stats)
speech = sum(s["speech_seconds"] for s in all_stats)
print(f"\n{len(all_stats)} audios, {total / 3600:.2f} h, speech {speech / 3600:.2f} h ({100 * speech / max(total, 1):.1f}%)")

## One audio in detail

In [ ]:
from IPython.display import Image, display

i = 0  # index from the list at the top
out_dir = out_root / "diarization" / wavs[i].stem / "nemotron"
print(json.dumps(json.loads((out_dir / "stats.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
display(Image(str(out_dir / "timeline.png")))